# 04 Embed And Pack Artifacts

Create BGE-M3 embedding JSONL artifacts, run offline retrieval smoke, and zip outputs for download.

This notebook keeps `state/`, `payloads/`, `embeddings/`, and `reports/` together so later sessions can resume from the same artifact bundle.

In [ ]:
import json, os, shutil, subprocess, sys, zipfile
from pathlib import Path

CORPUS_SLUG = 'tuvi-golden-corpus'
SCRIPTS_SLUG = 'tuvi-battu-scripts'

ALL_STRATEGIES = ['chunk_fixed_512', 'chunk_structure_parent_child', 'chunk_semantic_embedding_bge_m3']
SOURCE_IDS = ['TVKL', 'TVNL', 'TVHS', 'TVGM']
RUN_TAG = 'part_a'
PARTITION_MODE = 'strategy'
SELECTED_STRATEGIES = ['chunk_fixed_512', 'chunk_structure_parent_child']
SELECTED_SOURCES = list(SOURCE_IDS)
PREVIOUS_OUTPUT_SLUGS = []  # e.g. ['w3-local-outputs/w3_local_outputs_03_part_a'] or ['w3-03-part-a-output']
INPUT_ROOT_CANDIDATES = [
    Path('/kaggle/input/datasets/dinhbaobao'),
    Path('/kaggle/input'),
]

def resolve_input_root() -> Path:
    for candidate in INPUT_ROOT_CANDIDATES:
        if candidate.exists():
            return candidate
    return Path('/kaggle/input')

def resolve_input_path(value: str) -> Path:
    path = Path(value)
    if path.is_absolute():
        return path
    return INPUT_ROOT / path

INPUT_ROOT = resolve_input_root()
CORPUS_ROOT = resolve_input_path(CORPUS_SLUG)
SCRIPTS_ROOT = resolve_input_path(SCRIPTS_SLUG)
CORPUS_DIR = CORPUS_ROOT / 'benchmark' / 'tuvi_golden_dataset'
SCRIPTS_DIR = SCRIPTS_ROOT
if not CORPUS_DIR.exists():
    CORPUS_DIR = Path.cwd() / 'benchmark' / 'tuvi_golden_dataset'
if not SCRIPTS_DIR.exists():
    SCRIPTS_DIR = Path.cwd()

if PARTITION_MODE not in {'strategy', 'source'}:
    raise ValueError('PARTITION_MODE must be strategy or source')
ACTIVE_STRATEGIES = [item for item in ALL_STRATEGIES if item in SELECTED_STRATEGIES] if PARTITION_MODE == 'strategy' else list(ALL_STRATEGIES)
ACTIVE_SOURCES = [item for item in SOURCE_IDS if item in SELECTED_SOURCES] if PARTITION_MODE == 'source' else list(SOURCE_IDS)
if not ACTIVE_STRATEGIES:
    raise ValueError('No active strategies selected')
if not ACTIVE_SOURCES:
    raise ValueError('No active sources selected')

OUTPUT_ROOT = Path('/kaggle/working/w3_local_outputs')
os.environ['PYTHONDONTWRITEBYTECODE'] = '1'
REPORTS = OUTPUT_ROOT / 'reports'
STATE = OUTPUT_ROOT / 'state'
PAYLOADS = OUTPUT_ROOT / 'payloads'
EMBEDDINGS = OUTPUT_ROOT / 'embeddings'
for path in [REPORTS, STATE, PAYLOADS, EMBEDDINGS]:
    path.mkdir(parents=True, exist_ok=True)

def copy_tree_contents(source_dir: Path, destination_dir: Path) -> None:
    for child in source_dir.iterdir():
        target = destination_dir / child.name
        if child.is_dir():
            shutil.copytree(child, target, dirs_exist_ok=True)
        else:
            target.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(child, target)

def find_unpacked_artifact_roots(input_root: Path, expected_dir_name: str) -> list[Path]:
    named_dirs = []
    if input_root.is_dir() and input_root.name == expected_dir_name:
        named_dirs.append(input_root)
    if input_root.is_dir():
        named_dirs.extend(path for path in input_root.rglob(expected_dir_name) if path.is_dir())
    search_dirs = named_dirs or ([input_root] if input_root.is_dir() else [])
    artifact_roots = []
    for directory in search_dirs:
        if (directory / 'chunks').is_dir():
            artifact_roots.append(directory)
        artifact_roots.extend(path.parent for path in directory.rglob('chunks') if path.is_dir())
    unique_roots = []
    seen = set()
    for root in artifact_roots:
        resolved = root.resolve()
        if resolved not in seen:
            seen.add(resolved)
            unique_roots.append(root)
    return unique_roots

def restore_previous_outputs(slugs, expected_zip_name, expected_dir_name):
    restored = []
    search_roots = [resolve_input_path(slug) for slug in slugs] if slugs else [INPUT_ROOT]
    for input_root in search_roots:
        if not input_root.exists():
            raise FileNotFoundError(f'Previous output dataset not attached: {input_root}')
        zip_files = sorted(path for path in input_root.rglob(expected_zip_name) if path.is_file())
        artifact_roots = find_unpacked_artifact_roots(input_root, expected_dir_name)
        if zip_files:
            for zip_path in zip_files:
                print('Restoring previous output zip =', zip_path)
                with zipfile.ZipFile(zip_path) as archive:
                    archive.extractall(OUTPUT_ROOT)
                restored.append(str(zip_path))
            continue
        if artifact_roots:
            selected_root = artifact_roots[0]
            print('Restoring previous output folder =', selected_root)
            copy_tree_contents(selected_root, OUTPUT_ROOT)
            restored.append(str(selected_root))
            continue
        raise FileNotFoundError(f'No {expected_zip_name} or {expected_dir_name} found under {input_root}')
    return restored

RESTORED_OUTPUTS = restore_previous_outputs(PREVIOUS_OUTPUT_SLUGS, f'w3_local_outputs_03_{RUN_TAG}.zip', f'w3_local_outputs_03_{RUN_TAG}')
missing_inputs = []
for strategy in ACTIVE_STRATEGIES:
    if not (OUTPUT_ROOT / 'chunks' / strategy).exists():
        missing_inputs.append(str(OUTPUT_ROOT / 'chunks' / strategy))
    if not (PAYLOADS / strategy).exists():
        missing_inputs.append(str(PAYLOADS / strategy))
if missing_inputs:
    raise FileNotFoundError('Missing embedding/import-pack inputs. Attach notebook 03 artifact and set PREVIOUS_OUTPUT_SLUGS. Missing: ' + ', '.join(missing_inputs))

print('INPUT_ROOT =', INPUT_ROOT)
print('CORPUS_DIR =', CORPUS_DIR)
print('SCRIPTS_DIR =', SCRIPTS_DIR)
print('OUTPUT_ROOT =', OUTPUT_ROOT)
print('RESTORED_OUTPUTS =', RESTORED_OUTPUTS)
print(json.dumps({'run_tag': RUN_TAG, 'partition_mode': PARTITION_MODE, 'active_strategies': ACTIVE_STRATEGIES, 'active_sources': ACTIVE_SOURCES}, ensure_ascii=False, indent=2))

In [ ]:
for strategy in ACTIVE_STRATEGIES:
    for source in ACTIVE_SOURCES:
        cmd = [
            sys.executable, '-B', str(SCRIPTS_DIR / 'scripts' / 'embed_chunks.py'),
            '--chunks-input', str(OUTPUT_ROOT / 'chunks' / strategy),
            '--source-id', source,
            '--chunking-strategy', strategy,
            '--output', str(EMBEDDINGS / strategy / f'{source}_{strategy}_embeddings.jsonl'),
            '--summary-output', str(REPORTS / f'embed_{source}_{strategy}.json'),
            '--retrieval-smoke-output', str(REPORTS / f'retrieval_{source}_{strategy}.json'),
            '--state-output', str(STATE / f'{source}_{strategy}_embedding_state.json'),
            '--resume',
            '--embedding-slot', 'bge_m3',
            '--embedding-backend', 'local',
            '--model', 'BAAI/bge-m3',
            '--expected-dim', '1024',
            '--local-embedding-model', 'BAAI/bge-m3',
            '--local-embedding-batch-size', '16',
            '--vector-index-name', 'chunkVectorBgeM3',
        ]
        print('Running source/strategy =', source, strategy)
        subprocess.run(cmd, cwd=SCRIPTS_DIR, check=True)

print('Skipped strategies =', [item for item in ALL_STRATEGIES if item not in ACTIVE_STRATEGIES])
print('Skipped sources =', [item for item in SOURCE_IDS if item not in ACTIVE_SOURCES])

In [ ]:
for strategy in ACTIVE_STRATEGIES:
    for source in ACTIVE_SOURCES:
        embedding_path = EMBEDDINGS / strategy / f'{source}_{strategy}_embeddings.jsonl'
        state_path = STATE / f'{source}_{strategy}_embedding_state.json'
        print(source, strategy)
        print('  embedding =', embedding_path.exists(), embedding_path)
        print('  state     =', state_path.exists(), state_path)

archive = shutil.make_archive(f'/kaggle/working/w3_local_outputs_{RUN_TAG}', 'zip', OUTPUT_ROOT)
print('Wrote', archive)
print('Zip contains state/, payloads/, embeddings/, and reports/ for later resume/import.')